# Solution to p24

The audit separately checks graph validity, split use, and directional cross-attention semantics. For updating image tokens from graph facts, image tokens must supply queries while graph tokens supply keys and values, producing image-by-graph scores.

In [ ]:
from copy import deepcopy

trace = {
    "adjacency": ((1,1,0,0), (1,1,1,1), (0,1,1,0), (0,1,0,0)),
    "valid_nodes": (True, True, True, False),
    "train_ids": ("graph-train-00", "graph-train-01", "graph-train-02", "graph-train-03"),
    "heldout_ids": ("graph-heldout-00", "graph-heldout-01"),
    "optimizer_ids": ("graph-train-00", "graph-heldout-00"),
    "task": "update image tokens from graph facts",
    "q_source": "graph_tokens",
    "k_source": "image_tokens",
    "v_source": "image_tokens",
    "score_shape": ("B", "N_graph", "N_image"),
    "target_derived_edges": (),
}

def audit_cross_modal_sources(trace):
    expected = {
        "q_source":"image_tokens",
        "k_source":"graph_tokens",
        "v_source":"graph_tokens",
        "score_shape":("B","N_image","N_graph"),
        "split_disjoint":True,
        "binary_valid_node_masked_adjacency":True,
        "target_derived_edges":False,
    }
    violations = []
    adjacency = trace["adjacency"]
    valid = trace["valid_nodes"]
    n = len(valid)
    square = len(adjacency) == n and all(len(row) == n for row in adjacency)
    binary = square and all(value in (0,1) for row in adjacency for value in row)
    padded_clear = square and all(
        valid[i] or (all(adjacency[i][j] == 0 for j in range(n)) and all(adjacency[j][i] == 0 for j in range(n)))
        for i in range(n)
    )
    target_free = not bool(trace.get("target_derived_edges", ()))
    if not (binary and padded_clear and target_free):
        violations.append({
            "code":"invalid-padded-edge",
            "affected_semantics":"adjacency includes an invalid padded-node edge or a non-binary/target-derived edge",
            "repair":"zero every padded-node row and column, keep adjacency binary, and construct edges without targets",
        })
    heldout_optimizer = set(trace["optimizer_ids"]) & set(trace["heldout_ids"])
    split_disjoint = not heldout_optimizer and set(trace["train_ids"]).isdisjoint(trace["heldout_ids"])
    if not split_disjoint:
        violations.append({
            "code":"heldout-optimizer-id",
            "affected_semantics":"held-out examples influence optimizer updates",
            "repair":"restrict optimizer_ids to the immutable train_ids and reserve heldout_ids for evaluation",
        })
    directional = (
        trace.get("task") == "update image tokens from graph facts"
        and trace.get("q_source") == expected["q_source"]
        and trace.get("k_source") == expected["k_source"]
        and trace.get("v_source") == expected["v_source"]
        and tuple(trace.get("score_shape", ())) == expected["score_shape"]
    )
    if not directional:
        violations.append({
            "code":"reversed-qkv",
            "affected_semantics":"the trace updates graph tokens from images instead of image tokens from graph facts",
            "repair":"use image_tokens as Q, graph_tokens as K and V, and scores shaped (B,N_image,N_graph)",
        })
    return {"ok": not violations, "violations": tuple(violations), "expected": expected}

before = deepcopy(trace)
audit = audit_cross_modal_sources(trace)


### Answer check

In [ ]:
required_expected={
 'q_source':'image_tokens','k_source':'graph_tokens','v_source':'graph_tokens',
 'score_shape':('B','N_image','N_graph'),'split_disjoint':True,
 'binary_valid_node_masked_adjacency':True,'target_derived_edges':False,
}
assert trace==before
assert tuple(audit)==("ok","violations","expected")
assert audit["ok"] is False
assert audit["expected"]==required_expected
assert tuple(v["code"] for v in audit["violations"])==(
    "invalid-padded-edge","heldout-optimizer-id","reversed-qkv"
)
assert all(tuple(v)==("code","affected_semantics","repair") for v in audit["violations"])
valid_trace=deepcopy(trace)
valid_trace["adjacency"]=((1,1,0,0),(1,1,1,0),(0,1,1,0),(0,0,0,0))
valid_trace["optimizer_ids"]=valid_trace["train_ids"]
valid_trace["q_source"]="image_tokens"
valid_trace["k_source"]="graph_tokens"
valid_trace["v_source"]="graph_tokens"
valid_trace["score_shape"]=("B","N_image","N_graph")
valid_audit=audit_cross_modal_sources(valid_trace)
assert valid_audit["ok"] is True and valid_audit["violations"]==()
